<a href="https://colab.research.google.com/github/vanadhisivakumar-source/Machine-learning-projects/blob/main/non_parametric_weighted_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First, let's download a suitable dataset. I'll use the 'tips' dataset, which is a common dataset for demonstrating regression tasks, containing 'total_bill' and 'tip' information.

In [ ]:
import requests

# URL for the raw tips.csv file on GitHub
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"

# Define the local file path in the /tmp directory
file_path = "/tmp/tips.csv"

# Download the file
try:
    response = requests.get(url)
    response.raise_for_status() # Raise an exception for HTTP errors
    with open(file_path, 'wb') as f:
        f.write(response.content)
    print(f"Dataset downloaded successfully to {file_path}")
except requests.exceptions.RequestException as e:
    print(f"Error downloading the dataset: {e}")
except IOError as e:
    print(f"Error saving the dataset: {e}")

Now, let's implement the Locally Weighted Regression (LOWESS) algorithm with the corrected syntax and updated NumPy practices. I've also added comments to clarify the purpose of each section.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def kernel(point, xmat, k):
    """
    Calculates the weights for locally weighted regression using a Gaussian kernel.

    Parameters:
    point : numpy.ndarray
        The point for which to calculate weights.
    xmat : numpy.ndarray
        The feature matrix (design matrix).
    k : float
        The bandwidth parameter for the kernel.

    Returns:
    numpy.ndarray
        A diagonal matrix of weights.
    """
    m = np.shape(xmat)[0]
    weights = np.eye(m)
    for j in range(m):
        # Ensure point and xmat[j] are 1D arrays for subtraction
        diff = point - xmat[j]
        weights[j, j] = np.exp(diff @ diff.T / (-2.0 * k**2))
    return weights

def localWeight(point, xmat, ymat, k):
    """
    Performs a single weighted linear regression for a given point.

    Parameters:
    point : numpy.ndarray
        The point for which to make a prediction.
    xmat : numpy.ndarray
        The feature matrix (design matrix).
    ymat : numpy.ndarray
        The target variable matrix.
    k : float
        The bandwidth parameter for the kernel.

    Returns:
    numpy.ndarray
        The regression coefficients (beta).
    """
    wei = kernel(point, xmat, k)

    # Use @ for matrix multiplication and np.linalg.inv for inverse
    # Also, ensure dimensions are correct for matrix operations
    XT_W_X = xmat.T @ wei @ xmat

    # Add a small epsilon to the diagonal for numerical stability if XT_W_X is singular
    # or close to singular
    try:
        beta = np.linalg.inv(XT_W_X) @ xmat.T @ wei @ ymat
    except np.linalg.LinAlgError:
        # Handle singular matrix case - for example, return zeros or a default
        # In a real scenario, you might want to increase k or handle sparse data
        print("Warning: Singular matrix encountered. Returning zero coefficients.")
        beta = np.zeros(np.shape(xmat)[1]).reshape(-1, 1) # Return appropriate shape

    return beta

def localWeightRegression(xmat, ymat, k):
    """
    Applies locally weighted regression to all points in xmat.

    Parameters:
    xmat : numpy.ndarray
        The feature matrix (design matrix).
    ymat : numpy.ndarray
        The target variable matrix.
    k : float
        The bandwidth parameter for the kernel.

    Returns:
    numpy.ndarray
        The predicted y values.
    """
    m = np.shape(xmat)[0]
    ypred = np.zeros(m)
    for i in range(m):
        # xmat[i] is a row vector, needs to be treated as a matrix for localWeight
        ypred[i] = (xmat[i] @ localWeight(xmat[i], xmat, ymat, k)).item()
    return ypred.flatten() # Ensure ypred is a 1D array for plotting

# Load data points
data = pd.read_csv("/tmp/tips.csv")

# Extract 'total_bill' and 'tip' as numpy arrays
bill = np.array(data.total_bill)
tip = np.array(data.tip)

# Prepare data for regression:
# Add a column of ones for the intercept term
m = np.shape(bill)[0]
X = np.vstack([np.ones(m), bill]).T # Design matrix [1, total_bill]
y = tip.reshape(-1, 1)            # Reshape tip to a column vector

# Set the bandwidth parameter k
# This value often requires tuning based on the data
k = 0.5

# Perform locally weighted regression
ypred = localWeightRegression(X, y, k)

# Sort the data for proper plotting of the regression line
sort_indices = np.argsort(X[:, 1])
x_sorted = X[sort_indices]
ypred_sorted = ypred[sort_indices]

# Plot the results
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(1, 1, 1)
ax.scatter(bill, tip, color='green', label='Original Data')
ax.plot(x_sorted[:, 1], ypred_sorted, color='red', linewidth=3, label=f'LOWESS (k={k})')
plt.title('Locally Weighted Regression (LOWESS) on Tips Dataset')
plt.xlabel('Total Bill ($)')
plt.ylabel('Tip ($)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()